# MedVision AI - Interactive Research Lab

**Ported for High-Performance Compute (HPC) Environment**

This notebook contains the complete pipeline for MedVision AI, optimized for running on powerful GPU servers (e.g., NVIDIA DGX). validating `ResNet50`, `VGG16`, `VGG19`, and `CNN+CLAHE` models with interactive Grad-CAM visualization and RAG-based reporting.

**Suggested Port**: 11006

In [ ]:
# 1. Environment Setup & Imports
import os
import sys
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

# Ensure we use the GPU
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detected: {len(gpus)} device(s) available.")
    for gpu in gpus:
        print(f" - {gpu}")
else:
    print("⚠️ No GPU detected. Running on CPU (Inference might be slower).")

# Add current directory to path for local imports
sys.path.append(os.path.abspath(''))

In [ ]:
# 2. Configuration & Model Loading

# Define model paths (Adjust if your folder structure is different on the server)
MODEL_MAP = {
    "CNN+CLAHE": {"path": "models/cnn_clahe/model_cnn_clahe.keras", "layer": "conv_idx_2"},
    "ResNet50": {"path": "models/resnet50/model_resnet50.keras", "layer": "conv5_block3_out"},
    "VGG16": {"path": "models/vgg16/model_vgg16.keras", "layer": "block5_conv3"},
    "VGG19": {"path": "models/vgg19/model_vgg19.keras", "layer": "block5_conv4"}
}

def load_specific_model(model_name):
    if model_name not in MODEL_MAP:
        raise ValueError(f"Model {model_name} not found.")
    
    cfg = MODEL_MAP[model_name]
    path = cfg['path']
    
    if not os.path.exists(path):
        print(f"❌ Model file not found at: {path}")
        return None
        
    print(f"🔄 Loading {model_name} from {path}...")
    model = load_model(path)
    print(f"✅ {model_name} loaded successfully.")
    return model

In [ ]:
# 3. preprocessing & XAI Utilities

def apply_clahe(img):
    """Applies Contrast Limited Adaptive Histogram Equalization"""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_c = clahe.apply(img)
    return img_c / 255.0

def make_gradcam_heatmap(img_tensor, model, last_conv_layer_name):
    try:
        # Check for nested models (common in Transfer Learning)
        base_model = None
        base_layer_idx = -1
        for i, layer in enumerate(model.layers):
            if isinstance(layer, Model):
                base_model = layer
                base_layer_idx = i
                break
        
        if base_model:
            # Setup gradient model for nested architecture
            internal_grad_model = Model(
                inputs=[base_model.input],
                outputs=[base_model.get_layer(last_conv_layer_name).output, base_model.output]
            )
            
            with tf.GradientTape() as tape:
                x = model.layers[0](img_tensor)
                conv_outputs, base_output = internal_grad_model(x)
                x = base_output
                for i in range(base_layer_idx + 1, len(model.layers)):
                    x = model.layers[i](x)
                predictions = x
                class_index = tf.argmax(predictions[0])
                loss = predictions[:, class_index]
        else:
            # Standard flat architecture
            grad_model = Model(
                inputs=[model.inputs],
                outputs=[model.get_layer(last_conv_layer_name).output, model.output]
            )
            with tf.GradientTape() as tape:
                conv_outputs, predictions = grad_model(img_tensor)
                class_index = tf.argmax(predictions[0])
                loss = predictions[:, class_index]
                
        # Extract Gradients
        grads = tape.gradient(loss, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        
        heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
        heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
        
        return heatmap.numpy()
        
    except Exception as e:
        print(f"⚠️ Grad-CAM failed: {e}")
        return None

def overlay_gradcam(img, heatmap, alpha=0.4):
    img_rgb = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    heatmap_res = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255*heatmap_res), cv2.COLORMAP_JET)
    return cv2.addWeighted(img_rgb, 1-alpha, heatmap_color, alpha, 0)

In [ ]:
# 4. RAG & Reporting Setup
try:
    from rag_engine import RAGEngine
    from report_generator import ReportGenerator
    
    kb_path = 'knowledge_base.md'
    if os.path.exists(kb_path):
        print("🔄 Initializing RAG Engine...")
        report_gen = ReportGenerator(kb_path)
        print("✅ RAG Engine Ready.")
    else:
        print(f"❌ Knowledge base not found at {kb_path}")
        report_gen = None
except ImportError:
    print("❌ Could not import local RAG modules. Ensure rag_engine.py and report_generator.py are in this folder.")
    report_gen = None

In [ ]:
# 5. Interactive Diagnosis

def analyze_mammogram(image_path, model_name="CNN+CLAHE"):
    if not os.path.exists(image_path):
        print(f"❌ Image not found: {image_path}")
        return

    # 1. Load Model
    model = load_specific_model(model_name)
    if not model: return

    # 2. Load & Validate Image
    img_raw = cv2.imread(image_path)
    
    # Color Check (New Feature)
    hsv = cv2.cvtColor(img_raw, cv2.COLOR_BGR2HSV)
    if np.mean(hsv[:,:,1]) > 25:
        print("⚠️ WARNING: High color saturation detected. This might not be a mammogram.")
    
    img_gray = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img_gray, (128, 128))
    img_proc = apply_clahe(img)
    img_input = img_proc.reshape(1, 128, 128, 1)

    # 3. Predict
    preds = model.predict(img_input)
    pred_idx = np.argmax(preds[0])
    confidence = preds[0][pred_idx] * 100
    label = "Cancer" if pred_idx == 1 else "Non-Cancer"

    # 4. Grad-CAM
    heatmap = make_gradcam_heatmap(
        tf.convert_to_tensor(img_input, dtype=tf.float32),
        model, 
        MODEL_MAP[model_name]['layer']
    )
    if heatmap is not None:
        vis_img = overlay_gradcam(img_proc, heatmap)
    else:
        vis_img = cv2.cvtColor((img_proc*255).astype(np.uint8), cv2.COLOR_GRAY2BGR)

    # 5. RAG Report
    report = "RAG Engine not available."
    if report_gen:
        finding_text = "High activation in suspect region." if label == "Cancer" else "Diffuse background activation."
        report = report_gen.generate_report(label, confidence, finding_text)

    # 6. Visualization
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.title("Original CLAHE")
    plt.imshow(img_proc, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(f"Diagnosis: {label} ({confidence:.1f}%)\nGrad-CAM Visualization")
    plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    
    plt.show()
    
    display(Markdown(report))

In [ ]:
# TEST BLOCK
# Replace 'test_image.jpg' with your path once uploaded to Jupyter Lab
# analyze_mammogram("test_image.jpg", "ResNet50")